In [38]:
import pandas as pd
import numpy as np
from datetime import timedelta
import random

In [39]:
import os
print(os.listdir('D:/Drug-data-analysis/dataset'))

['pharmacy-inventory-optimization.xlsx', 'pharmacy-inventory.csv']


In [40]:


df = pd.read_csv(r'D:/Drug-data-analysis/dataset/pharmacy-inventory.csv', encoding='latin1')
print(df)

    pharmacy_id               pharmacy_name     address_street   address_city  \
0         PH001           CityCare Pharmacy  128 Market Avenue  San Francisco   
1         PH002             EcoPharm Berlin  12 Alexanderplatz         Berlin   
2         PH003            Central Pharmacy    478 Main Street        Chicago   
3         PH004     Remote Village Pharmacy        9 Nuuk Road           Nuuk   
4         PH005    Family Pharmacy Brisbane    34 Queen Street       Brisbane   
..          ...                         ...                ...            ...   
195       PH102  Greenland Village Pharmacy     67 Aurora Lane           Nuuk   
196       PH103       Sapphire Care Chemist   23 Sapphire Road         Sydney   
197       PH104        Central Rx Solutions        412 King St         London   
198       PH105        Lakefront Apothecary  88 Lakeside Drive         Zurich   
199       PH106           Maple Leaf Health   355 Maple Avenue        Toronto   

    address_state address_p

In [33]:
df.columns


Index(['pharmacy_id', 'pharmacy_name', 'address_street', 'address_city',
       'address_state', 'address_postal_code', 'address_country', 'product_id',
       'product_name', 'product_category', 'product_manufacturer',
       'stock_level', 'stock_threshold_min', 'stock_threshold_max',
       'average_daily_demand', 'predicted_demand_next_30_days',
       'last_reorder_date', 'next_reorder_date_predicted',
       'reorder_quantity_suggested', 'stockout_risk_flag',
       'excess_inventory_flag', 'last_stockout_date', 'notes'],
      dtype='object')

In [35]:
print("Original dataset shape:", df.shape)

Original dataset shape: (200, 23)


Expand dataset (200 → 800 rows)


In [16]:
multiplier = 4  # 200 × 4 = 800 rows approx
df_large = pd.concat([df] * multiplier, ignore_index=True)

Add variation to numeric columns

In [18]:
if 'stock_level' in df_large.columns:
    df_large['stock_level'] = df_large['stock_level'] + np.random.randint(-30, 30, size=len(df_large))

if 'average_daily_demand' in df_large.columns:
    df_large['average_daily_demand'] = df_large['average_daily_demand'] + np.random.randint(-5, 5, size=len(df_large))

# Update predicted demand and reorder quantity
df_large['predicted_demand_next_30_days'] = df_large['average_daily_demand'] * 30

# Recalculate reorder quantities
df_large['reorder_quantity_suggested'] = np.where(
    df_large['stock_level'] < df_large['stock_threshold_min'],
    np.random.randint(100, 300, size=len(df_large)),
    np.random.randint(10, 80, size=len(df_large))
)

Update stock risk and excess flags

In [19]:
df_large['stockout_risk_flag'] = df_large['stock_level'] < df_large['stock_threshold_min']
df_large['excess_inventory_flag'] = >df_large['stock_level'] > df_large['stock_threshold_max']

Global Pharmacy Address Generator

In [36]:
countries = {
    'India': {
        'cities': ['Mumbai', 'Pune', 'Delhi', 'Chennai', 'Hyderabad', 'Bengaluru', 'Ahmedabad', 'Kolkata', 'Nagpur', 'Nashik'],
        'state': 'Maharashtra'
    },
    'USA': {
        'cities': ['New York', 'Los Angeles', 'Chicago', 'San Francisco', 'Boston', 'Houston', 'Seattle'],
        'state': 'CA'
    },
    'UK': {
        'cities': ['London', 'Manchester', 'Birmingham', 'Leeds', 'Liverpool'],
        'state': 'England'
    },
    'Germany': {
        'cities': ['Berlin', 'Munich', 'Hamburg', 'Frankfurt', 'Cologne'],
        'state': 'Germany'
    },
    'Australia': {
        'cities': ['Sydney', 'Melbourne', 'Brisbane', 'Perth', 'Adelaide'],
        'state': 'Australia'
    },
    'Canada': {
        'cities': ['Toronto', 'Vancouver', 'Montreal', 'Calgary', 'Ottawa'],
        'state': 'Canada'
    }
}

# 🟢 70% India, 30% other countries
weights = [0.7, 0.1, 0.05, 0.05, 0.05, 0.05]

df_large['address_country'] = np.random.choice(
    list(countries.keys()),
    size=len(df_large),
    p=weights  # India gets higher probability
)

# Assign city/state based on selected country
df_large['address_city'] = df_large['address_country'].apply(lambda c: np.random.choice(countries[c]['cities']))
df_large['address_state'] = df_large['address_country'].apply(lambda c: countries[c]['state'])

# Random streets and postal codes
streets = ['MG Road', 'Station Road', 'FC Road', 'Main Bazaar', 
           'Link Road', 'Residency Road', 'Market Street', 'Church Street']

df_large['address_street'] = np.random.choice(streets, size=len(df_large))
df_large['address_postal_code'] = np.random.randint(400000, 700000, size=len(df_large))

Generate realistic, drug-specific notes

In [37]:
def generate_note(row):
    drug = str(row.get('product_name', 'This medicine'))
    category = str(row.get('product_category', '')).lower()

    if row.get('stockout_risk_flag', False):
        return f"Low stock for {drug}; urgent reorder needed due to high demand."
    elif row.get('excess_inventory_flag', False):
        return f"Recent overstock of {drug}; monitor consumption to avoid wastage."
    elif 'supplement' in category:
        return f"Steady demand for {drug}; recommend moderate restock next month."
    elif 'antihypertensive' in category:
        return f"Essential hypertension drug {drug}; maintain steady supply levels."
    elif 'antibiotic' in category:
        return f"Regular prescriptions for {drug}; ensure timely procurement."
    elif 'painkiller' in category:
        return f"Increased usage of {drug} observed; reorder before stock runs out."
    elif 'antacid' in category:
        return f"Stock of {drug} stable; reorder as per routine schedule."
    elif 'antidiabetic' in category:
        return f"Consistent need for {drug}; reorder based on monthly averages."
    elif 'antihistamine' in category:
        return f"Rising seasonal demand for {drug}; monitor stock closely."
    else:
        return f"Normal demand pattern for {drug}; continue current stock cycle."

df_large['notes'] = df_large.apply(generate_note, axis=1)

huffle & limit to 700 rows

In [23]:
df_large = df_large.sample(700, random_state=42).reset_index(drop=True)
print("Expanded dataset shape:", df_large.shape)

Expanded dataset shape: (700, 23)


Save as CSV and Excel

In [41]:
df_large.to_csv('final_pharmacy_inventory.csv', index=False)
df_large.to_excel('final_pharmacy_inventory.xlsx', index=False)

print("\n✅ Done! Files saved as:")
print("   → final_pharmacy_inventory.csv")

print("\n🌍 Global dataset ready: 700 rows, 7 countries, realistic pharmacy details & smart notes.")


✅ Done! Files saved as:
   → final_pharmacy_inventory.csv

🌍 Global dataset ready: 700 rows, 7 countries, realistic pharmacy details & smart notes.
